# FULL OWOD CHAIN V2 — the research plan's own score, on the detector

**Select a T4 GPU, then Runtime -> Run all, then leave it. It is resumable —
expect to press Run all again after a disconnect.** Everything lands on Drive
under `results/full_owod_chain_v2/`.

Pre-registration: **`docs/full_owod_v2_protocol.md`**, frozen before this
notebook's first trajectory. Cell [4/12] compares this notebook's parameter cell
against that document **as values** and refuses to continue if they disagree.

## What is new, and why

The committed chains ran arms that are **not** the research plan's method:
three static rankings, two farthest-first traversals in DINOv2 space, and a
cluster-quota allocator. The plan's own equation

    s(x) = U(x) + lambda*D(x) + gamma*w(c_hat(x))*coh(x)

existed only in `owl/scoring.py`, which runs on the frozen CPU pool and never
trains a detector. This session puts it on the GPU path and makes the
2026-08-25 consultation's open questions **configurable axes** instead of fixed
choices:

| axis | what moves | where |
|---|---|---|
| `D(x)` | labelled novelty against a pool that **grows**, intra-batch diversity, or both | `owl/active_selection/research_score.py` |
| `coh(x)` | binary DBSCAN gate (`coh` in {0,1}), the plan's continuous form, or open | same |
| `w(c_hat(x))` | cluster rarity from the **same** partition the gate produced | same |
| annotation | a real **per-box** policy, with ignore that is not background | `owl/supervision.py` |
| replay | `m_c ~ n_c^alpha`, re-derived per task | `owl/replay.py` |
| acquisition | the budget in mini-rounds, score recomputed between them | `research_score.select(rounds=...)` |

## What this is NOT

* **Not a re-run of Benchmark V1 or of the t10 chain.** It writes to its own
  results directory, its `CycleConfig` fingerprint differs, and a workspace
  collision is refused rather than blended. No committed number is touched.
* **Not the published S-OWODB benchmark.** This is the repository's own
  controlled chain, one new class per task. No number here may be compared
  against a published S-OWODB number. Running the published S-OWODB and
  M-OWODB sequences is open work, recorded in the protocol's section 14.
* **Not a tuned method.** `lambda = 0.2` and `gamma = 0.5` are the values
  `owl.scoring` froze in 2026-08 before any endpoint was read; `min_samples`
  comes from the answer budget and `eps` from the candidate set's own geometry.
  Nothing was chosen because it won on the seed-0 results already on disk.

## Before the first run

`OWL_COMMIT` in cell [1/12] is a **placeholder**. Commit the V2 code, then paste
the 40-character SHA in. Cell [1/12] fails immediately if you do not — which is
deliberate: a notebook that runs against an unpinned checkout produces a result
nobody can reproduce.

In [ ]:
# [1/12] PARAMETERS — the human-readable config, and the immutable identity
# ============================== PARAMETERS ==============================
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

# Before anything imports torch. Expandable segments let a freed block be
# reused at a different size instead of fragmenting the reserve, which is what
# turns "enough free memory in total" into an OOM. Memory management only; it
# cannot change a number.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ---------------------------------------------------------------- the config --
#
# Every value here is compared against docs/full_owod_v2_protocol.md in cell
# [9/12], as values. Editing one without editing the protocol document fails the
# preflight rather than quietly running a different experiment.

PHASE = "A"                 # "A" = protocol validation, short chain, one seed
                            # "B" = the full comparison, t1 -> t10, three seeds

TASK_END = 5                # 5 for Phase A, 10 for Phase B
SEEDS = [0]                 # [0] for Phase A, [0, 1, 2] for Phase B

ANNOTATION_POLICY = "known_plus_selected_ignore_rest"
IGNORE_MECHANISM = "pixel_suppression"
REPLAY_MODE = "uniform"     # none | uniform | proportional | tail_aware
REPLAY_REFRESH = "fixed"    # fixed | per_task
COHERENCE_MODE = "dbscan_binary"    # none | continuous | dbscan_binary
DIVERSITY_MODE = "combined"         # none | labeled_novelty | batch_diversity | combined
ACQUISITION_BATCH_SIZE = 100        # answers per mini-round; rounds = ceil(B/batch)
ANSWER_BUDGET = 300                 # oracle answers per task

# The arms, in the registry's own execution order regardless of how they are
# written here. Cell [9/12] checks this set against the phase's pre-registered
# arms in the protocol document, so a reported run cannot quietly run a subset.
#
#   Phase A: ("random", "entropy", "research_v2")
#   Phase B: ("random", "entropy", "distribution_aware_iterative_v1",
#             "research_v2_plan", "research_v2")
SESSION_ARMS = ("random", "entropy", "research_v2")

# One Run all is allowed this much GPU time. The launcher stops BETWEEN tasks
# when it runs out, which keeps the checkpoint lineage intact.
TIME_BUDGET_MINUTES = 1200

# --------------------------------------------------------------- the identity --
OWL_REPOSITORY = "https://github.com/gubiczam/owod-active.git"
# REPLACE THIS. Commit the V2 code, then paste its 40-character SHA here. A
# notebook cannot contain the SHA of the commit containing it, so this is set by
# hand once, and tools/validate_notebook_freshness.py checks the pin by
# executing the pinned code rather than by comparing trees.
OWL_COMMIT = "REPLACE_WITH_THE_40_CHAR_SHA_OF_THE_V2_COMMIT"
PROB_REPOSITORY = "https://github.com/gubiczam/PROB.git"
PROB_COMMIT = "4c66be1a52cad9360e09c729e9134aba8fe0b531"

DRIVE_ROOT = "/content/drive/MyDrive/OWL"
CHECKPOINT_RELATIVE = "checkpoints/SOWODB/t1.pth"
FEATURES_RELATIVE = "features"
# Its own directory. Benchmark V1 and the t10 chain do not occupy it, and
# nothing here can overwrite either.
RESULTS_RELATIVE = "results/full_owod_chain_v2"

DATA_ROOT = "/content/data/OWOD"

N_TASKS = TASK_END
SESSION_STARTED = time.monotonic()

assert len(PROB_COMMIT) == 40, "pin the full 40-char PROB SHA"
assert len(OWL_COMMIT) == 40, (
    "OWL_COMMIT is still the placeholder. Commit the V2 code and paste its "
    "40-character SHA into cell [1/12]. Running against an unpinned checkout "
    "produces a result nobody can reproduce, so this stops here rather than "
    "later.")
print("OWL commit :", OWL_COMMIT)
print("PROB commit:", PROB_COMMIT)
print(f"phase      : {PHASE}  |  t1 -> t{TASK_END}  |  seeds {SEEDS}")
print(f"arms       : {SESSION_ARMS}")
print(f"annotation : {ANNOTATION_POLICY} / {IGNORE_MECHANISM}")
print(f"replay     : {REPLAY_MODE} / {REPLAY_REFRESH}")
print(f"score      : D={DIVERSITY_MODE}  coh={COHERENCE_MODE}")
print(f"acquisition: {ANSWER_BUDGET} answers per task in "
      f"{-(-ANSWER_BUDGET // ACQUISITION_BATCH_SIZE)} rounds of "
      f"{ACQUISITION_BATCH_SIZE}")

In [ ]:
# [2/12] Mount Drive and prove the persistent root is writable
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
_probe = DRIVE / ".benchmark_v1_write_probe"
_probe.write_text("ok", encoding="utf-8")
assert _probe.read_text(encoding="utf-8") == "ok"
_probe.unlink()

FEATURES = DRIVE / FEATURES_RELATIVE
RESULTS = DRIVE / RESULTS_RELATIVE
CHECKPOINT = DRIVE / CHECKPOINT_RELATIVE
RESULTS.mkdir(parents=True, exist_ok=True)
print("Drive writable:", DRIVE)
print("results ->", RESULTS)


In [ ]:
# [3/12] Pin OWL exactly, install its declared dependencies, import fresh code
def _checked(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)


def _capture(command, **kwargs):
    return _checked(command, capture_output=True, **kwargs).stdout.strip()


def _normalise_git_url(value):
    value = value.strip().removesuffix(".git").rstrip("/")
    if value.startswith("git@github.com:"):
        value = "https://github.com/" + value.split(":", 1)[1]
    return value


def ensure_pinned_checkout(path, repository, commit):
    path = Path(path)
    expected = _normalise_git_url(repository)
    if path.exists():
        assert (path / ".git").is_dir(), f"Refusing non-git path: {path}"
        origin = _normalise_git_url(_capture(["git", "remote", "get-url", "origin"], cwd=path))
        assert origin == expected, f"Refusing unexpected origin at {path}: {origin}"
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        _checked(["git", "clone", "--filter=blob:none", "--no-checkout", repository, str(path)])
    _checked(["git", "fetch", "--depth", "1", "origin", commit], cwd=path)
    _checked(["git", "reset", "--hard", commit], cwd=path)
    _checked(["git", "clean", "-fdx"], cwd=path)
    actual = _capture(["git", "rev-parse", "HEAD"], cwd=path)
    assert actual == commit, f"{path}: expected {commit}, got {actual}"
    return path


ROOT = ensure_pinned_checkout(Path("/content/owod-active"), OWL_REPOSITORY, OWL_COMMIT)
_checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q",
          "-e", f"{ROOT}[plots]"])

for _name in [n for n in sys.modules if n == "owl" or n.startswith("owl.")]:
    del sys.modules[_name]
sys.path.insert(0, str(ROOT))

from owl import bridge, evaluation_subset, metrics, protocol, runner
from owl.active_selection import arms as arm_registry
from owl.active_selection import benchmark as bm
from owl.active_selection import budget as annotation_budget
from owl.active_selection import coverage, population, semantic

# Named checks, so a stale OWL_COMMIT reports which API it is missing instead of
# failing as an AttributeError four cells later.
_required = {
    "runner.CycleConfig.budget_unit": "budget_unit" in {
        f.name for f in __import__("dataclasses").fields(runner.CycleConfig)},
    "run_chain.selector": "selector" in __import__("inspect").signature(
        runner.run_chain).parameters,
    "bm.check_protocol": hasattr(bm, "check_protocol"),
    "bm.make_selector": hasattr(bm, "make_selector"),
    "bm.cycle_config": hasattr(bm, "cycle_config"),
    "arms.ORDER": hasattr(arm_registry, "ORDER"),
    "coverage.kcenter_greedy": hasattr(coverage, "kcenter_greedy"),
    "population.p2_reference": hasattr(population, "p2_reference"),
    "semantic.cached": hasattr(semantic, "cached"),
    "budget.cost_function": hasattr(annotation_budget, "cost_function"),
    # Two things a symbol check cannot see, both of which a stale pin would
    # silently lack: the ledger column that says what PROB was actually handed,
    # and the fail-closed guard that stops a resumed chain from restarting a
    # task from the anchor and reporting a sequential result it never produced.
    "runner.boxes_trained_on": "boxes_trained_on" in __import__("inspect").getsource(
        runner.run_chain),
    "runner.lineage_guard": "break the checkpoint lineage" in __import__(
        "inspect").getsource(runner.run_chain),
    # Proposed-v2's surface. A stale pin lacking any of these must say which.
    "arms.proposed_v2": "proposed_v2" in arm_registry.ARMS,
    "arms.ranked_positions": hasattr(arm_registry, "ranked_positions"),
    "arms.informative": getattr(
        arm_registry.ARMS.get("proposed_v2"), "informative", False),
    "arms.reference_scope": getattr(
        arm_registry.ARMS.get("proposed_v2"), "reference_scope", "") == "trajectory",
    "bm.KILL_RULE": hasattr(bm, "KILL_RULE"),
    "bm.PROVENANCE": hasattr(bm, "PROVENANCE"),
    "semantic.release": hasattr(semantic, "release"),
}
assert all(_required.values()), {k: v for k, v in _required.items() if not v}
# Properties, not a literal copy of the registry: a hardcoded tuple has to be
# hand-edited every time an arm is added, and when it is not, Run all dies here.
assert arm_registry.ORDER[:5] == (
    "random", "admissibility", "proposed", "entropy", "coreset",
), arm_registry.ORDER
assert set(arm_registry.ORDER) == set(arm_registry.ARMS), arm_registry.ORDER
assert set(SESSION_ARMS) <= set(arm_registry.ARMS), SESSION_ARMS
assert bm.N_TASKS == 4, "V1 stays frozen at four tasks; this session passes --n-tasks"
assert bm.ANSWER_BUDGET_PER_TASK == 3000
# Generic in N_TASKS, because V2 runs a five-task Phase A and a ten-task Phase B
# out of the same notebook. A chain of N tasks declares N-1 classes and is scored
# on its own shared split, which must not be V1's.
assert [t.name for t in bm.chain(N_TASKS)] == [
    f"t{_i}" for _i in range(1, N_TASKS + 1)]
assert len(bm.declared_classes(N_TASKS)) == N_TASKS - 1, bm.declared_classes(N_TASKS)
assert evaluation_subset.shared_test_set_name(N_TASKS) \
    != evaluation_subset.shared_test_set_name(bm.N_TASKS), "split must differ"
# The V2 surface. A stale pin lacking any of these must say which, rather than
# failing later with a bare TypeError on an unexpected keyword.
_v2_required = {
    "arms.research_v2": "research_v2" in arm_registry.ARMS,
    "arms.score_spec": getattr(
        arm_registry.ARMS.get("research_v2"), "score_spec", None) is not None,
    "arms.picks": "picks" in {f.name for f in __import__("dataclasses").fields(
        arm_registry.ArmSelection)},
    "arms.anchor_boxes": "anchor_boxes" in {
        f.name for f in __import__("dataclasses").fields(arm_registry.ArmSelection)},
    "research_score": hasattr(
        __import__("owl.active_selection.research_score", fromlist=["x"]), "select"),
    "supervision.policies": hasattr(
        __import__("owl.supervision", fromlist=["x"]), "POLICIES"),
    "replay.MODES": hasattr(__import__("owl.replay", fromlist=["x"]), "MODES"),
    "v2.phase": hasattr(
        __import__("owl.active_selection.v2", fromlist=["x"]), "phase"),
    "runner.annotation_policy": "annotation_policy" in {
        f.name for f in __import__("dataclasses").fields(runner.CycleConfig)},
    "runner.data_root": "data_root" in __import__("inspect").signature(
        runner.run_chain).parameters,
    "cycle_config.v2_axes": "annotation_policy" in __import__(
        "inspect").signature(bm.cycle_config).parameters,
    "make_selector.rounds": "rounds" in __import__("inspect").signature(
        bm.make_selector).parameters,
    "metrics.WI_and_AOSE": "WI08" in __import__("inspect").getsource(
        __import__("owl.metrics", fromlist=["x"]).task_row),
}
assert all(_v2_required.values()), {
    k: v for k, v in _v2_required.items() if not v}
print("chain:", [t.name for t in bm.chain()])
print("budget:", bm.ANSWER_BUDGET_PER_TASK, "oracle answers per task,",
      bm.CANDIDATE_IMAGES_PER_TASK, "candidate images")
OWL_SHA = _capture(["git", "rev-parse", "HEAD"], cwd=ROOT)
print("OWL ready:", OWL_SHA, "from", ROOT)

In [ ]:
# [4/12] The pre-registration, checked as VALUES before anything expensive runs
from owl.active_selection import v2 as v2_protocol

_preset = v2_protocol.phase(PHASE)
CONFIGURATION = v2_protocol.Configuration(
    task_end=TASK_END,
    seeds=tuple(SEEDS),
    annotation_policy=ANNOTATION_POLICY,
    ignore_mechanism=IGNORE_MECHANISM,
    replay_mode=REPLAY_MODE,
    replay_refresh=REPLAY_REFRESH,
    coherence_mode=COHERENCE_MODE,
    diversity_mode=DIVERSITY_MODE,
    acquisition_batch_size=ACQUISITION_BATCH_SIZE,
    answer_budget=ANSWER_BUDGET,
    arms=tuple(SESSION_ARMS),
    phase=PHASE,
)
_agreement = v2_protocol.check(CONFIGURATION)
print("protocol check:", _agreement.statement())
assert _agreement.agrees, _agreement.disagreements

# The arms this phase pre-registered. A subset is allowed — a session that ran
# out of runtime finishes the rest — but an arm the phase never declared is not.
assert set(SESSION_ARMS) <= set(_preset.arms), (
    f"{sorted(set(SESSION_ARMS) - set(_preset.arms))} are not phase {PHASE}'s "
    f"pre-registered arms {_preset.arms}. Adding an arm to a running protocol "
    "is how an arm comes to be chosen by its numbers.")
if set(SESSION_ARMS) != set(_preset.arms):
    print("NOTE: running a SUBSET of phase", PHASE, "—",
          sorted(set(_preset.arms) - set(SESSION_ARMS)), "not in this session.")

# Phase A is deliberately shorter and single-seed; Phase B is not allowed to be.
if PHASE == "B":
    assert TASK_END == _preset.task_end, (TASK_END, _preset.task_end)
    assert sorted(SEEDS) == sorted(_preset.seeds), (SEEDS, _preset.seeds)
else:
    assert TASK_END <= _preset.task_end, (
        f"Phase A is protocol validation on a short chain (t1 -> t"
        f"{_preset.task_end}); a longer one is Phase B and needs its arms and "
        "its seeds.")

# An arm is a frozen object. A notebook variable may not silently redefine one,
# so a disagreement between the cell's modes and the arm's registered ScoreSpec
# is named rather than resolved.
_mismatch = CONFIGURATION.score_mismatch()
assert not _mismatch, _mismatch

# Registry order, not the cell's: a session that runs out of runtime must
# complete a prefix that was fixed in advance.
SESSION_ARMS = CONFIGURATION.arm_order()
print("arms          :", SESSION_ARMS, "(registry execution order)")
print("launcher flags:", " ".join(CONFIGURATION.launcher_flags()))
for _name in SESSION_ARMS:
    _spec = arm_registry.ARMS[_name]
    _score = _spec.score_spec
    print(f"  {_name:34s} {_spec.kind:10s} "
          + (f"D={_score.diversity_mode} coh={_score.coherence_mode} "
             f"w={_score.rarity_mode} lam={_score.lambda_diversity} "
             f"gam={_score.gamma_rarity}" if _score else _spec.description[:58]))

In [ ]:
# [5/12] PREFLIGHT — is the pinned PROB source actually there?
#
# A Method V3 overnight run died in the next cell on
#
#     git clone --filter=blob:none --no-checkout .../PROB.git   ->  exit status 128
#
# after Drive was mounted and OWL was installed, and a bare 128 does not say
# whether the URL is wrong, the pinned commit is gone, or a shared Colab egress
# address was rate-limited for a minute. Guessing at that is how a frozen
# detector gets quietly swapped for a convenient one, so the question is
# answered here — before pip, before the CUDA kernel build — and the answer
# names the URL and the SHA.
# Runtime -> Run all executes cells in order and this holds automatically. It is
# checked anyway because the failure it replaces is a bare `NameError: name
# 'bridge' is not defined`, which says nothing about what to do — and a Colab
# user who runs one cell to "just check something" gets exactly that.
if "bridge" not in globals():
    raise RuntimeError(
        "cell [3/10] has not run in this kernel, so `bridge` does not exist. "
        "It is the cell that pins OWL and imports the package. Use "
        "Runtime -> Run all rather than running cells individually."
    )

PROB_REMOTE = bridge.verify_remote_commit(PROB_REPOSITORY, PROB_COMMIT)
print("PROB repository :", PROB_REMOTE["repository"], "— reachable")
print("PROB commit     :", PROB_REMOTE["commit"], "— present on the server")
print("PROB branch     :", PROB_REMOTE["branch"], "->", PROB_REMOTE["branch_head"])
print("pin is a ref tip:", PROB_REMOTE["pin_is_ref_tip"],
      "| branch still points at the pin:", PROB_REMOTE["branch_points_at_commit"])
print("probe attempts  :", PROB_REMOTE["attempts_used"])
if not PROB_REMOTE["branch_points_at_commit"]:
    print("NOTE: the branch has moved on since the pin. The pin is honoured "
          "anyway — that is what a pin is for.")


In [ ]:
# [6/12] Pin and validate the reviewed PROB bridge; build the optional CUDA kernel
# An exact local checkout is accepted as-is: origin must be the pinned
# repository and HEAD must be the pinned SHA, so what is accepted is
# byte-identical to what a clone would have produced. That makes a second
# attempt in the same session free, and it is the offline recovery — a mirror of
# that SHA copied in from Drive is scientifically the same run.
_PROB_PATH = Path("/content/PROB")
if bridge.local_checkout_matches(_PROB_PATH, PROB_REPOSITORY, PROB_COMMIT):
    PROB = _PROB_PATH
    print("PROB already checked out at the pinned commit; no network needed.")
else:
    PROB = None
    for _attempt in range(1, 4):
        try:
            PROB = ensure_pinned_checkout(_PROB_PATH, PROB_REPOSITORY, PROB_COMMIT)
            break
        except subprocess.CalledProcessError as _error:
            # A failed clone can leave a partial directory behind, which the next
            # attempt would then treat as an existing checkout. Clear it unless it
            # is a real git repository.
            if not (_PROB_PATH / ".git").is_dir():
                shutil.rmtree(_PROB_PATH, ignore_errors=True)
            print(f"PROB checkout attempt {_attempt}/3 failed: {_error}")
            if _attempt == 3:
                raise
            time.sleep(10)
    assert PROB is not None

# PROB's 2022 requirements file pins packages that have no Python 3.13 wheels
# (notably scikit-image 0.19.2 and pandas 1.5.1). The bridge does not import
# scikit-image, notebook, or ipdb. Install only its runtime imports, without
# replacing Colab's matched torch/torchvision/numpy stack. pycocotools stays
# at PROB's exact 2.0.5 pin: Cython generates the C source omitted by its sdist.
assert sys.version_info[:2] == (3, 13), sys.version
def distribution_version(distribution):
    probe = subprocess.run(
        [sys.executable, "-c",
         f"from importlib.metadata import version; print(version({distribution!r}))"],
        capture_output=True, text=True, check=False)
    return probe.stdout.strip() if probe.returncode == 0 else None


def module_available(module):
    return subprocess.run(
        [sys.executable, "-c", f"import {module}"],
        capture_output=True, text=True, check=False).returncode == 0


PROB_COMPAT_INSTALLED = []
if distribution_version("einops") != "0.5.0" or not module_available("einops"):
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              "einops==0.5.0"])
    PROB_COMPAT_INSTALLED.append("einops==0.5.0")

if (distribution_version("pycocotools") != "2.0.5"
        or not module_available("pycocotools")):
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              "Cython==3.1.3"])
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--no-build-isolation",
              "--no-deps", "--force-reinstall", "pycocotools==2.0.5"])
    PROB_COMPAT_INSTALLED.append("pycocotools==2.0.5")

# These imports are required transitively by main_open_world/engine. Keep a
# compatible Colab package when one is already importable; install a pinned
# Python-3.13 wheel only when it is absent or broken. WandB is disabled by the
# reviewed bridge and therefore cannot affect training or evaluation.
compatibility_wheels = {
    "wandb": "wandb==0.18.7",
    "pandas": "pandas==2.3.2",
    "seaborn": "seaborn==0.13.2",
    "tqdm": "tqdm==4.67.1",
}
missing_wheels = [spec for module, spec in compatibility_wheels.items()
                  if not module_available(module)]
if missing_wheels:
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              *missing_wheels])
    PROB_COMPAT_INSTALLED.extend(missing_wheels)

assert distribution_version("einops") == "0.5.0" and module_available("einops")
assert (distribution_version("pycocotools") == "2.0.5"
        and module_available("pycocotools"))


def pip_check():
    return subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        capture_output=True, text=True, check=False)


# Current Colab carries IPython metadata that requires Jedi while omitting
# Jedi itself. Repair only that observed metadata conflict, using a universal
# wheel with explicit Python 3.13 support, then require the complete package
# environment to pass the same check used by the final preflight.
bootstrap_package_probe = pip_check()
_package_conflicts = bootstrap_package_probe.stdout + bootstrap_package_probe.stderr
_missing_ipython_jedi = (
    "requires jedi, which is not installed" in _package_conflicts.lower()
    and "ipython " in _package_conflicts.lower()
)
if _missing_ipython_jedi:
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              "jedi==0.19.2"])
    PROB_COMPAT_INSTALLED.append("jedi==0.19.2")
    bootstrap_package_probe = pip_check()
if bootstrap_package_probe.returncode != 0:
    print(bootstrap_package_probe.stdout + bootstrap_package_probe.stderr)
    raise RuntimeError("Python package consistency check failed after bootstrap repair")
print("Bootstrap package consistency: PASS")
runtime_probe = subprocess.run(
    [sys.executable, "-c",
     "import numpy, torch, torchvision, scipy, sklearn, PIL, matplotlib, pandas, seaborn, tqdm, wandb; "
     "from einops import rearrange; from pycocotools.coco import COCO; "
     "import main_open_world; from datasets.coco import make_coco_transforms; "
     "from datasets.torchvision_datasets.open_world import OWDetection; "
     "from engine import evaluate; from models import build_model"],
    cwd=PROB, capture_output=True, text=True, check=False)
if runtime_probe.returncode != 0:
    print(runtime_probe.stdout)
    print(runtime_probe.stderr)
    raise RuntimeError("Pinned PROB failed its Python runtime import probe")
print("PROB runtime imports: PASS; installed:", PROB_COMPAT_INSTALLED or "nothing")

# pycocotools 2.0.5 predates NumPy 2.0. Exercise PROB's own evaluator
# wrapper with the two removed aliases it needs, instead of accepting an
# import-only success. The aliases are confined to this fresh subprocess.
coco_smoke_code = r'''
import numpy as np
import torch
if "float" not in np.__dict__:
    np.float = float
if "NPY_OWNDATA" not in np.__dict__:
    np.NPY_OWNDATA = 4
from pycocotools.coco import COCO
from datasets.coco_eval import CocoEvaluator
coco = COCO()
coco.dataset = {
    "info": {}, "licenses": [],
    "images": [{"id": 1, "width": 32, "height": 32}],
    "categories": [{"id": 1, "name": "object", "supercategory": "object"}],
    "annotations": [{"id": 1, "image_id": 1, "category_id": 1,
                     "bbox": [4.0, 5.0, 10.0, 11.0], "area": 110.0, "iscrowd": 0}],
}
coco.createIndex()
evaluator = CocoEvaluator(coco, ("bbox",))
evaluator.update({1: {"boxes": torch.tensor([[4.0, 5.0, 14.0, 16.0]]),
                      "scores": torch.tensor([0.99]), "labels": torch.tensor([1])}})
evaluator.synchronize_between_processes()
evaluator.accumulate()
evaluator.summarize()
assert float(evaluator.coco_eval["bbox"].stats[0]) > 0.99
print("PROB pycocotools COCOeval smoke: PASS")
'''
coco_smoke = subprocess.run([sys.executable, "-c", coco_smoke_code], cwd=PROB,
                            capture_output=True, text=True, check=False)
if coco_smoke.returncode != 0:
    print(coco_smoke.stdout)
    print(coco_smoke.stderr)
    raise RuntimeError("Pinned pycocotools failed PROB's functional COCOeval smoke test")
print(coco_smoke.stdout.splitlines()[-1])


def run_json_probe(code, marker, *, cwd):
    probe = subprocess.run(
        [sys.executable, "-c", code], cwd=cwd,
        capture_output=True, text=True, check=False,
    )
    rows = [line.removeprefix(marker) for line in probe.stdout.splitlines()
            if line.startswith(marker)]
    if not rows:
        return {
            "probe_ok": False,
            "returncode": probe.returncode,
            "error": (probe.stderr or probe.stdout).strip() or "probe produced no result",
        }
    payload = json.loads(rows[-1])
    payload["returncode"] = probe.returncode
    return payload


# Deliberately diagnostic only: unlike PROB, this fresh interpreter does not
# import torch before loading the extension. On Colab that can fail to resolve
# PyTorch shared libraries even when PROB's real import and dispatch work.
raw_msda_probe_code = r"""
import importlib
import json
try:
    importlib.invalidate_caches()
    extension = importlib.import_module("MultiScaleDeformableAttention")
    payload = {"ok": True, "path": getattr(extension, "__file__", None), "error": None}
except BaseException as error:
    payload = {"ok": False, "path": None,
               "error": f"{type(error).__name__}: {error}"}
print("OWOD_RAW_MSDA_PROBE=" + json.dumps(payload, sort_keys=True))
"""
RAW_MSDA = run_json_probe(
    raw_msda_probe_code, "OWOD_RAW_MSDA_PROBE=", cwd=PROB.parent)

# Authoritative pre-build/post-build probe: import through the exact wrapper and
# downstream module used by PROB training. The pinned wrapper imports torch
# before the extension and the downstream module copies this boolean for dispatch.
prob_msda_probe_code = r"""
import importlib
import importlib.metadata
import json
import platform
import sys
from pathlib import Path
import einops
import matplotlib
import numpy
import pandas
import PIL
import pycocotools
import scipy
import sklearn
import torch
import torchvision

after_torch = {"ok": False, "path": None, "error": None}
try:
    importlib.invalidate_caches()
    extension = importlib.import_module("MultiScaleDeformableAttention")
    after_torch = {"ok": True, "path": getattr(extension, "__file__", None), "error": None}
except BaseException as error:
    after_torch["error"] = f"{type(error).__name__}: {error}"

payload = {
    "probe_ok": False,
    "python": platform.python_version(),
    "executable": sys.executable,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "torch_cuda": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "pillow": PIL.__version__,
    "matplotlib": matplotlib.__version__,
    "pandas": pandas.__version__,
    "einops": importlib.metadata.version("einops"),
    "pycocotools": importlib.metadata.version("pycocotools"),
    "extension_after_torch": after_torch,
}
try:
    from models.ops.functions import ms_deform_attn_func as msda_func
    from models.ops.modules import ms_deform_attn as msda_module
    expected_root = Path.cwd().resolve()
    wrapper_path = Path(msda_func.__file__).resolve()
    downstream_path = Path(msda_module.__file__).resolve()
    assert wrapper_path.is_relative_to(expected_root), wrapper_path
    assert downstream_path.is_relative_to(expected_root), downstream_path
    available = bool(msda_func.MSDA_AVAILABLE)
    assert bool(msda_module.MSDA_AVAILABLE) == available
    payload.update({
        "probe_ok": True,
        "available": available,
        "backend": "compiled" if available else "PyTorch fallback",
        "wrapper_path": str(wrapper_path),
        "downstream_path": str(downstream_path),
        "extension_path": (getattr(msda_func.MSDA, "__file__", None)
                           if available else None),
        "error": None,
    })
except BaseException as error:
    payload["error"] = f"{type(error).__name__}: {error}"
print("OWOD_PROB_MSDA_PROBE=" + json.dumps(payload, sort_keys=True))
"""


def probe_prob_msda():
    return run_json_probe(
        prob_msda_probe_code, "OWOD_PROB_MSDA_PROBE=", cwd=PROB)


PREBUILD_PROB_MSDA = probe_prob_msda()
_msda_fingerprint = json.dumps({
    "prob": PROB_COMMIT,
    "python": PREBUILD_PROB_MSDA.get("python", sys.version),
    "torch": PREBUILD_PROB_MSDA.get("torch"),
    "torch_cuda": PREBUILD_PROB_MSDA.get("torch_cuda"),
    "gpu": PREBUILD_PROB_MSDA.get("gpu"),
}, sort_keys=True)
MSDA_BUILD_MARKER = (
    PROB.parent / ".owod-active-cache" /
    f"msda-build-{hashlib.sha256(_msda_fingerprint.encode()).hexdigest()[:16]}.json"
)
MSDA_BUILD_ATTEMPTED = False
MSDA_BUILD_RETURN_CODE = None
if PREBUILD_PROB_MSDA.get("available") is not True:
    if MSDA_BUILD_MARKER.is_file():
        print("Skipping a previously built-but-unused MSDA extension for this exact runtime:",
              MSDA_BUILD_MARKER)
    else:
        if not module_available("ninja"):
            _checked([sys.executable, "-m", "pip", "install",
                      "--disable-pip-version-check", "-q", "ninja"])
        MSDA_BUILD_ATTEMPTED = True
        build = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "--no-build-isolation", "--no-deps", "--force-reinstall", "."],
            cwd=PROB / "models" / "ops", text=True,
            capture_output=True, check=False,
        )
        MSDA_BUILD_RETURN_CODE = build.returncode
        if build.returncode != 0:
            print("WARNING: optional CUDA extension build failed; the full CUDA smoke "
                  "must prove PROB's PyTorch fallback.")
            print("\n".join((build.stdout + "\n" + build.stderr).splitlines()[-20:]))

POSTBUILD_PROB_MSDA = probe_prob_msda()
print("Raw extension import:", "PASS" if RAW_MSDA.get("ok") else "FAIL")
print("Raw extension path:", RAW_MSDA.get("path") or "unavailable")
if RAW_MSDA.get("error"):
    print("Raw extension error:", RAW_MSDA["error"])
print("PROB MSDA_AVAILABLE:", POSTBUILD_PROB_MSDA.get("available"))
print("PROB wrapper path:", POSTBUILD_PROB_MSDA.get("wrapper_path", "unavailable"))
print("PROB extension path:", POSTBUILD_PROB_MSDA.get("extension_path", "unavailable"))
if RAW_MSDA.get("ok") != POSTBUILD_PROB_MSDA.get("available"):
    print("MSDA diagnostic disagreement explained: the raw probe loads the extension "
          "before torch; pinned PROB imports torch first, then binds the extension "
          "inside models.ops.functions.ms_deform_attn_func. Only PROB's downstream "
          "dispatch is authoritative.")

# Run the real PROB builder and training loss on CUDA. This observes the exact
# MSDeformAttn branch taken by the model and requires that branch to participate
# in forward and backward before any experiment evaluation or training can run.
prob_smoke_code = r"""
import json
import sys
import tempfile
from pathlib import Path
import numpy as np
import torch
from PIL import Image
if "bool" not in np.__dict__:
    np.bool = np.bool_
import main_open_world
from datasets.coco import make_coco_transforms
from datasets.open_world_eval import voc_eval
from datasets.torchvision_datasets.open_world import OWDetection
from models import build_model
from models.ops.functions import ms_deform_attn_func as msda_func
from models.ops.modules import ms_deform_attn as msda_module
assert torch.cuda.is_available(), "CUDA is unavailable to the real PROB smoke test"
PROB_MSDA_AVAILABLE = bool(msda_func.MSDA_AVAILABLE)
assert bool(msda_module.MSDA_AVAILABLE) == PROB_MSDA_AVAILABLE
assert Path(msda_func.__file__).resolve().is_relative_to(Path.cwd().resolve())
assert Path(msda_module.__file__).resolve().is_relative_to(Path.cwd().resolve())
dispatch_counts = {"compiled": 0, "fallback": 0}
if PROB_MSDA_AVAILABLE:
    original_apply = msda_module.MSDeformAttnFunction.apply
    class ObservedCompiledDispatch:
        @staticmethod
        def apply(*arguments):
            dispatch_counts["compiled"] += 1
            return original_apply(*arguments)
    msda_module.MSDeformAttnFunction = ObservedCompiledDispatch
else:
    original_fallback = msda_module.ms_deform_attn_core_pytorch
    def observed_fallback(*arguments, **keywords):
        dispatch_counts["fallback"] += 1
        return original_fallback(*arguments, **keywords)
    msda_module.ms_deform_attn_core_pytorch = observed_fallback
args = main_open_world.get_args_parser().parse_args([])
args.device = "cuda"
args.dataset = "OWDETR"
args.PREV_INTRODUCED_CLS = 0
args.CUR_INTRODUCED_CLS = 20
args.num_classes = 81
args.model_type = "prob"
args.wandb_project = ""
args.wandb_name = ""
args.batch_size = 1
args.num_workers = 0
with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    (root / "Annotations").mkdir()
    (root / "JPEGImages").mkdir()
    (root / "ImageSets" / "OWDETR").mkdir(parents=True)
    image_id = "000000000001"
    (root / "ImageSets" / "OWDETR" / "smoke_val.txt").write_text(image_id + "\n")
    Image.new("RGB", (32, 32), (0, 0, 0)).save(root / "JPEGImages" / f"{image_id}.jpg")
    annotation = ("<annotation><filename>000000000001.jpg</filename>"
                  "<size><width>32</width><height>32</height><depth>3</depth></size>"
                  "<object><name>aeroplane</name><difficult>0</difficult>"
                  "<bndbox><xmin>1</xmin><ymin>1</ymin><xmax>20</xmax><ymax>20</ymax>"
                  "</bndbox></object></annotation>")
    xml_path = root / "Annotations" / f"{image_id}.xml"
    xml_path.write_text(annotation)
    dataset = OWDetection(args, root, image_set="smoke_val", dataset="OWDETR",
                          transforms=make_coco_transforms("smoke_val"))
    image, target = dataset[0]
    assert image.shape[0] == 3 and target["labels"].tolist() == [0]
    _, _, ap, *_ = voc_eval(
        [f"{image_id} 0.99 1 1 20 20"], [str(xml_path)], [image_id],
        "aeroplane", known_classes=["aeroplane"])
    assert float(ap) > 0.99
model, criterion, postprocessors, _ = build_model(args, mode="prob")
checkpoint_path = Path(sys.argv[1]) if sys.argv[1] else None
if checkpoint_path is not None:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    assert isinstance(checkpoint, dict) and isinstance(checkpoint.get("epoch"), int)
    state = checkpoint.get("model", checkpoint)
    model_state = model.state_dict()
    compatible_keys = [name for name, value in state.items()
                       if name in model_state and torch.is_tensor(value)
                       and value.shape == model_state[name].shape]
    assert compatible_keys, "T1 checkpoint has no compatible detector parameters"
    incompatible = model.load_state_dict(state, strict=False)
    assert torch.equal(model.state_dict()[compatible_keys[0]].cpu(),
                       state[compatible_keys[0]].cpu())
    print("T1 checkpoint parsed:", checkpoint["epoch"], len(compatible_keys),
          len(incompatible.missing_keys), len(incompatible.unexpected_keys))
model.to("cuda").train()
outputs = model([torch.rand(3, 64, 64, device="cuda")])
required = {"pred_logits", "pred_boxes", "pred_obj", "pred_features", "aux_outputs"}
assert required <= set(outputs)
targets = [{"labels": torch.tensor([0], device="cuda"),
            "boxes": torch.tensor([[0.5, 0.5, 0.25, 0.25]], device="cuda")}]
losses = criterion(outputs, targets)
weighted = sum(losses[name] * criterion.weight_dict[name]
               for name in losses if name in criterion.weight_dict)
assert torch.isfinite(weighted)
weighted.backward()
results = postprocessors["bbox"](outputs, torch.tensor([[64, 64]], device="cuda"))
assert len(results) == 1 and torch.isfinite(results[0]["boxes"]).all()
torch.cuda.synchronize()
chosen = "compiled" if PROB_MSDA_AVAILABLE else "PyTorch fallback"
assert dispatch_counts["compiled" if PROB_MSDA_AVAILABLE else "fallback"] > 0
assert dispatch_counts["fallback" if PROB_MSDA_AVAILABLE else "compiled"] == 0
print("OWOD_MSDA_RESULT=" + json.dumps({
    "available": PROB_MSDA_AVAILABLE,
    "backend": chosen,
    "dispatch_counts": dispatch_counts,
}, sort_keys=True))
print("MSDA backend:", chosen)
print("PROB CUDA model/loss/evaluator smoke: PASS")
"""
checkpoint_for_smoke = Path(DRIVE_ROOT) / CHECKPOINT_RELATIVE
smoke_checkpoint = str(checkpoint_for_smoke) if checkpoint_for_smoke.is_file() else ""
prob_smoke = subprocess.run(
    [sys.executable, "-c", prob_smoke_code, smoke_checkpoint],
    cwd=PROB, capture_output=True, text=True, check=False)
if prob_smoke.returncode != 0:
    print(prob_smoke.stdout)
    print(prob_smoke.stderr)
    print("Environment preflight: FAIL")
    raise RuntimeError("Pinned PROB failed its real CUDA model/evaluator smoke test")
_smoke_rows = [line.removeprefix("OWOD_MSDA_RESULT=")
               for line in prob_smoke.stdout.splitlines()
               if line.startswith("OWOD_MSDA_RESULT=")]
if not _smoke_rows:
    print(prob_smoke.stdout)
    print("Environment preflight: FAIL")
    raise RuntimeError("Real PROB smoke passed without an authoritative MSDA result")
MSDA_SMOKE_RESULT = json.loads(_smoke_rows[-1])
PROB_MSDA_AVAILABLE = bool(MSDA_SMOKE_RESULT["available"])
PROB_MSDA_BACKEND = MSDA_SMOKE_RESULT["backend"]
assert PROB_MSDA_BACKEND == ("compiled" if PROB_MSDA_AVAILABLE else "PyTorch fallback")
if (MSDA_BUILD_ATTEMPTED and MSDA_BUILD_RETURN_CODE == 0
        and not PROB_MSDA_AVAILABLE):
    MSDA_BUILD_MARKER.parent.mkdir(parents=True, exist_ok=True)
    MSDA_BUILD_MARKER.write_text(json.dumps({
        "fingerprint": _msda_fingerprint,
        "backend": PROB_MSDA_BACKEND,
        "reason": "extension built but PROB selected and fully verified fallback",
    }, indent=2), encoding="utf-8")
PROB_SHA = _capture(["git", "rev-parse", "HEAD"], cwd=PROB)
assert PROB_SHA == PROB_COMMIT
ENVIRONMENT_PREFLIGHT_OK = True
runtime = POSTBUILD_PROB_MSDA
print("=" * 60)
print("OWOD ENVIRONMENT PREFLIGHT")
print("=" * 60)
for label, value in (
    ("Runtime Python", runtime.get("python")),
    ("Torch", runtime.get("torch")),
    ("Torchvision", runtime.get("torchvision")),
    ("Torch CUDA", runtime.get("torch_cuda")),
    ("CUDA available", runtime.get("cuda_available")),
    ("GPU", runtime.get("gpu")),
    ("NumPy", runtime.get("numpy")),
    ("SciPy", runtime.get("scipy")),
    ("sklearn", runtime.get("sklearn")),
    ("Pillow", runtime.get("pillow")),
    ("matplotlib", runtime.get("matplotlib")),
    ("pandas", runtime.get("pandas")),
    ("einops", runtime.get("einops")),
    ("pycocotools", runtime.get("pycocotools")),
    ("OWL SHA", OWL_SHA),
    ("PROB SHA", PROB_SHA),
    ("Raw MSDA extension import", "PASS" if RAW_MSDA.get("ok") else "FAIL"),
    ("PROB MSDA_AVAILABLE", PROB_MSDA_AVAILABLE),
    ("MSDA backend", PROB_MSDA_BACKEND),
    ("PROB CUDA model/loss/evaluator smoke", "PASS"),
    ("Environment preflight", "PASS"),
):
    print(f"{label}: {value}")
print("=" * 60)



In [ ]:
# [7/12] What this session reads from Drive, and what it will compute
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


REF_T1 = FEATURES / "ref_t1_dinov2_vitb14_cap1000_v1.npz"
_needs_reference = any(arm_registry.ARMS[a].needs_semantic for a in SESSION_ARMS)

missing = [str(CHECKPOINT)] if not CHECKPOINT.is_file() else []
if _needs_reference and not REF_T1.is_file():
    missing.append(str(REF_T1))
assert not missing, (
    "Missing from Drive: " + str(missing) + ". t1.pth is PROB's published "
    "S-OWODB checkpoint; ref_t1_dinov2_vitb14_cap1000_v1.npz is the frozen "
    "balanced task-1 semantic reference produced by the Method V2 notebook, and "
    "the coverage arms measure distance to it.")

CHECKPOINT_SHA = sha256(CHECKPOINT)
print("checkpoint :", CHECKPOINT, f"({CHECKPOINT.stat().st_size / 1e6:.0f} MB)")
print("            sha256", CHECKPOINT_SHA)
if _needs_reference:
    print("ref_t1     :", REF_T1.name, f"({REF_T1.stat().st_size / 1e6:.0f} MB) — reused, not recomputed")
print()
print("COMPUTED this session: one DINOv2 pass per coverage arm per task, over "
      "that task's own fresh candidate pool. The frozen Method V2 crop is "
      "reused unchanged (owl.semantic_features); the backbone is never "
      "fine-tuned. Roughly 24,000 crops per task for a gated arm.")


In [ ]:
# [8/12] Build the data root: committed annotations, the shared split, the pixels
def _streamed(command, transcript=None):
    """Run a step, showing its output as it happens, and fail loudly.

    ``capture_output=True`` hides the traceback of the step that failed, which is
    the one thing a 3 a.m. Run all must not do. Everything expensive is streamed
    instead, and the exit code is asserted afterwards.
    """

    print("+", " ".join(map(str, command)))
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in process.stdout:
        print(line.rstrip(), flush=True)
        lines.append(line)
    code = process.wait()
    if transcript is not None:
        Path(transcript).parent.mkdir(parents=True, exist_ok=True)
        with Path(transcript).open("a", encoding="utf-8") as handle:
            handle.write(f"\n=== {time.strftime('%Y-%m-%dT%H:%M:%S')} ===\n")
            handle.writelines(lines)
    assert code == 0, f"{command[1]} exited {code}; its output is above"
    return "".join(lines)


_streamed([sys.executable, str(ROOT / "tools" / "prepare_full_owod_benchmark.py"),
           "--data-root", DATA_ROOT, "--n-tasks", str(N_TASKS)])

DATA = Path(DATA_ROOT)
# `shared_test_set_name(N_TASKS)`, not `SHARED_TEST_SET`: the latter is
# Benchmark V1's four-task name, and `prepare_full_owod_benchmark.py
# --n-tasks N` writes ONLY the split for N. On a fresh /content this cell
# asserted the existence of a file nothing had written -- while cell [3]
# asserts in the same notebook that the two names must DIFFER.
assert (DATA / "Annotations").is_dir() and (DATA / "JPEGImages").is_dir()
assert (DATA / "ImageSets" / "OWDETR" /
        f"{evaluation_subset.shared_test_set_name(N_TASKS)}.txt").is_file()
print("data root ready:", DATA)


In [ ]:
# [9/12] PREFLIGHT — what is frozen and what this run costs, printed BEFORE any
#                    training starts.
print("annotation :", ANNOTATION_POLICY)
print("            PROB has NO ignore channel — owl.supervision reads the pinned")
print("            source and says so — and", IGNORE_MECHANISM, "is how this run")
print("            delivers one. 'drop' is the honest name for what the pipeline")
print("            did before: the ignored region becomes a background target.")
print("cost       : the price of an image is a property of the policy —")
print("            full_image charges max(1, objects on it), the other two")
print("            charge 1 — so the answer budget moves with the policy and")
print("            images_opened is reported as the outcome, not equalised.")
print("replay     :", REPLAY_MODE, "/", REPLAY_REFRESH,
      f"| M={bm.REPLAY_OBJECTS} exemplar objects, identical for every arm")
print("training   :", f"{bm.EPOCHS} epochs, lr {bm.LEARNING_RATE}, "
      f"batch {bm.BATCH_SIZE}")
print("rounds     :", -(-ANSWER_BUDGET // ACQUISITION_BATCH_SIZE),
      "— the acquisition score is recomputed between them against a labelled")
print("            pool that grew; the DETECTOR is not retrained between rounds.")

print("\nPROVENANCE — this session runs arms that are NOT pre-registered for V1:")
for _line in bm.PROVENANCE:
    print("  *", _line)
for _line in bm.REPORTING:
    print(" !", _line)

_plan = _streamed([sys.executable, str(ROOT / "tools" / "plan_full_owod_benchmark.py"),
                   "--json", str(RESULTS / "plan.json")])

# The launcher's flags, checked before the overnight cell rather than inside it.
_help = subprocess.run(
    [sys.executable, str(ROOT / "tools" / "run_full_owod_benchmark.py"), "--help"],
    check=True, capture_output=True, text=True).stdout
for _flag in ("--prob-root", "--data-root", "--checkpoint", "--ref-t1", "--out",
              "--seeds", "--arms", "--time-budget-minutes", "--n-tasks",
              "--annotation-policy", "--ignore-mechanism", "--replay-mode",
              "--replay-refresh", "--acquisition-batch-size", "--answer-budget"):
    assert _flag in _help, f"the pinned launcher has no {_flag}"
print("\nlauncher flags present. Ready.")

In [ ]:
# [10/12] Run or resume. THIS IS THE LONG CELL — so EXPECT TO RUN THIS NOTEBOOK
#         MORE THAN ONCE. It resumes: a finished trajectory is skipped and an
#         interrupted one restarts at the task it died on, so pressing Run all
#         again after a disconnect is correct and loses nothing.
#
# HOW RESUME WORKS, since a disconnect depends on it. One workspace per
# (arm, seed). A task is COMPLETE when it has both `state.json` and
# `metrics.json`; such a task is skipped and its accumulated state — which
# images were opened, which labels were banked, which exemplars are held, and
# the boxes the oracle was asked about — is restored from `state.json` rather
# than recomputed. So a disconnect costs the task in flight and nothing else.
#
# WHAT THIS SESSION COSTS. Evaluation is the expensive half, not training: the
# shared reduced split is scored after every task of every arm, and
# `detections=True` costs a second forward pass, because the plan's headline
# endpoint — tail U-Recall against oracle cost — cannot come from the aggregate
# the evaluator prints. TIME_BUDGET_MINUTES is deliberately below a full
# session: the launcher stops BETWEEN tasks, so the last task or two will be
# left INCOMPLETE on the first pass. Run all again.
#
# Output is streamed, never captured, so a failure at 3 a.m. shows its traceback.
_elapsed = (time.monotonic() - SESSION_STARTED) / 60.0
_budget = max(TIME_BUDGET_MINUTES - _elapsed, 1.0)
print(f"{_elapsed:.0f} min of setup; handing {_budget:.0f} min to the launcher.\n")

_streamed([
    sys.executable, str(ROOT / "tools" / "run_full_owod_benchmark.py"),
    "--prob-root", str(PROB),
    "--data-root", DATA_ROOT,
    "--checkpoint", str(CHECKPOINT),
    "--ref-t1", str(REF_T1),
    "--out", str(RESULTS),
    "--time-budget-minutes", f"{_budget:.0f}",
    "--n-tasks", str(N_TASKS),
    "--seeds", *[str(_s) for _s in SEEDS],
    "--arms", *SESSION_ARMS,
    "--annotation-policy", ANNOTATION_POLICY,
    "--ignore-mechanism", IGNORE_MECHANISM,
    "--replay-mode", REPLAY_MODE,
    "--replay-refresh", REPLAY_REFRESH,
    "--acquisition-batch-size", str(ACQUISITION_BATCH_SIZE),
    "--answer-budget", str(ANSWER_BUDGET),
], transcript=RESULTS / "session_log.txt")

_manifest = json.loads((RESULTS / "manifest.json").read_text(encoding="utf-8"))
_status = {row["trajectory"]: row["status"] for row in _manifest["trajectories"]}
print("\n", _status)
assert not _manifest.get("dry_run"), "this manifest is from a stubbed run"

# The V2 configuration, written next to the results so a table can never be read
# without knowing which axes produced it.
(RESULTS / "v2_configuration.json").write_text(
    json.dumps(CONFIGURATION.as_dict(), indent=2), encoding="utf-8")

In [ ]:
# [11/12] The tables, the CSVs and the figures
_streamed([sys.executable, str(ROOT / "tools" / "summarize_full_owod_benchmark.py"),
           "--results", str(RESULTS)], transcript=RESULTS / "summary_log.txt")
_streamed([sys.executable, str(ROOT / "tools" / "plot_full_owod_benchmark.py"),
           "--results", str(RESULTS)])

_summary = json.loads((RESULTS / "summary.json").read_text(encoding="utf-8"))
_complete = [t for t in _summary["trajectories"] if t["status"] == "COMPLETE"]
_other = [t for t in _summary["trajectories"] if t["status"] != "COMPLETE"]

print("\n" + "=" * 78)
print(f"{len(_complete)} of {len(_summary['trajectories'])} trajectories complete")
for _row in _other:
    print("  NOT COMPLETE:", _row["trajectory"], _row["status"],
          _row.get("error", ""))
if _other:
    print("\nRe-run this notebook to continue them. They resume; nothing is lost.")

# The candidate-level trail the consultation asked for: per taken candidate,
# D_labeled, D_batch, the final D, w, coh, the cluster id, and whether the point
# was core, border or noise. One file per (arm, seed, task).
_logs = sorted(RESULTS.glob("*/t*/candidate_log.csv"))
print(f"\ncandidate logs: {len(_logs)} files")
for _path in _logs[:3]:
    print("  ", _path.relative_to(RESULTS))

print("\nEverything is on Drive:", RESULTS)
print("  manifest.json, summary.json, v2_configuration.json,")
print("  per_task_metrics.csv, supervision_cost.csv, acquisition.csv,")
print("  chain_summary.csv, forgetting.csv, per_class_ap.csv, contrasts.csv,")
print("  annotation_efficiency.csv, */t*/candidate_log.csv, plots/")
print(f"\nsession wall clock: {(time.monotonic() - SESSION_STARTED) / 60.0:.0f} min")

# [12/12] Phase A is protocol validation, not an endpoint comparison

Read these before drawing any conclusion. Phase A exists to show that the
machinery does what `docs/full_owod_v2_protocol.md` says, and the only thing it
may do to an arm is remove it for a **mechanical** reason (section 13): a
trajectory that did not complete, a cost ledger that does not reconcile, or a
coherence gate that degenerated. A low AP is a result to report, never grounds
for removal.

| file | what must hold |
|---|---|
| `supervision_cost.csv` | `objects_supervised + objects_ignored + objects_banked == objects_labelled`, and `known_boxes_reused > 0` from t3 on — that is the free supervision the middle policy exists for |
| `*/t*/candidate_log.csv` | `coh` takes **both** 0 and 1; `cluster_status` contains core **and** border **and** noise; `D_batch` moves within a round; `D == (D_labeled + D_batch)/2` under `combined` |
| `results_*.csv` (`round_log`) | `reference_rows` grows from round to round — that is the labelled pool feeding back |
| `per_task_metrics.csv` | `WI08` and `A_OSE` are present, not blank |
| `acquisition.csv` | `acquired_tail_objects` per answer, which is the detector-free endpoint and survives a failed trajectory |

**The primary endpoint is `U_Recall_tail` against `oracle_cost_so_far`**, and it
is read at Phase B, at t10, over three seeds — not here. A seed-0 difference is
a direction, not an effect.